In [1]:
import tensorflow as tf
import os, random, shutil
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

In [2]:
DATASET_DIR = "Dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 4
SEED = 42

# TRAIN (70%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# VAL (15%)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# SPLIT VAL & TEST (50:50 dari 30%)
val_size = int(0.5 * tf.data.experimental.cardinality(val_test_ds).numpy())
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Kelas:", class_names)


Found 1193 files belonging to 7 classes.
Using 836 files for training.
Found 1193 files belonging to 7 classes.
Using 357 files for validation.
Kelas: ['Antraknosa', 'Bercak Daun Cercospora', 'Buah Cabai Sehat', 'Busuk Buah', 'Daun Cabai Sehat', 'Penyakit Kuning Keriting', 'objek tidak dikenal']


In [3]:
# =====================================================
# 1. KONFIGURASI TRAINING
# =====================================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 4
EPOCHS = 30
SEED = 42

DATASET_DIR = "Dataset"  # folder utama

# =====================================================
# 2. LOAD DATASET (AUTO SPLIT)
# =====================================================

# TRAIN (70%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# VAL + TEST (30%)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# SPLIT VAL & TEST → 15% : 15%
val_size = int(0.5 * tf.data.experimental.cardinality(val_test_ds).numpy())
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Kelas:", class_names)
print("Jumlah kelas:", num_classes)


Found 1193 files belonging to 7 classes.
Using 836 files for training.
Found 1193 files belonging to 7 classes.
Using 357 files for validation.
Kelas: ['Antraknosa', 'Bercak Daun Cercospora', 'Buah Cabai Sehat', 'Busuk Buah', 'Daun Cabai Sehat', 'Penyakit Kuning Keriting', 'objek tidak dikenal']
Jumlah kelas: 7


In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)


In [6]:
# =====================================================
# 4. MODEL EfficientNetV2-S
# =====================================================
base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # data sedikit → freeze total

model = Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 224, 224, 3)       0         
                                                                 
 efficientnetv2-s (Function  (None, 7, 7, 1280)        20331360  
 al)                                                             
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout_1 (Dropout)         (None, 128)              

In [7]:
# =====================================================
# 5. CALLBACKS
# =====================================================
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model_efficientnetv2s.h5",
                    monitor="val_accuracy",
                    save_best_only=True)
]


In [8]:
# =====================================================
# 6. TRAINING
# =====================================================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/30
209/209 [==============================] - ETA: 0s - loss: 1.8571 - accuracy: 0.2907

c:\Users\Person\anaconda3\envs\hasil\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


209/209 [==============================] - 70s 270ms/step - loss: 1.8571 - accuracy: 0.2907 - val_loss: 1.3261 - val_accuracy: 0.6444
Epoch 2/30
209/209 [==============================] - 55s 262ms/step - loss: 1.4487 - accuracy: 0.4856 - val_loss: 1.0149 - val_accuracy: 0.7222
Epoch 3/30
209/209 [==============================] - 56s 265ms/step - loss: 1.2005 - accuracy: 0.5873 - val_loss: 0.8476 - val_accuracy: 0.7389
Epoch 4/30
209/209 [==============================] - 55s 265ms/step - loss: 1.0762 - accuracy: 0.6208 - val_loss: 0.7475 - val_accuracy: 0.7667
Epoch 5/30
209/209 [==============================] - 56s 269ms/step - loss: 0.9884 - accuracy: 0.6459 - val_loss: 0.6498 - val_accuracy: 0.7778
Epoch 6/30
209/209 [==============================] - 56s 267ms/step - loss: 0.8730 - accuracy: 0.7022 - val_loss: 0.5844 - val_accuracy: 0.7944
Epoch 7/30
209/209 [==============================] - 56s 265ms/step - loss: 0.7994 - accuracy: 0.7237 - val_loss: 0.5370 - val_accuracy: 0.8

In [9]:
# =====================================================
# 7. EVALUASI
# =====================================================
test_loss, test_acc = model.evaluate(test_ds)
print(f"Akurasi Test Set: {test_acc*100:.2f}%")


45/45 [==============================] - 10s 194ms/step - loss: 0.3296 - accuracy: 0.8814
Akurasi Test Set: 88.14%


In [10]:
# =====================================================
# 8. SIMPAN MODEL
# =====================================================
model.save("model_cabai_efficientnetv2s_final")
print("Model berhasil disimpan")

INFO:tensorflow:Assets written to: model_cabai_efficientnetv2s_final\assets


INFO:tensorflow:Assets written to: model_cabai_efficientnetv2s_final\assets


Model berhasil disimpan


In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(1, len(acc) + 1)

plt.figure()
plt.plot(epochs, acc, label='Training Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training dan Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#Confusion Matrix
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# =====================================================
# 7. PREDIKSI DATA TEST
# =====================================================
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = np.argmax(preds, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)



In [ ]:
# =====================================================
# 8. CLASSIFICATION REPORT
# =====================================================
print("\n=== Classification Report ===")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))
